In [5]:
import os
from dotenv import load_dotenv
from langgraph.graph import add_messages,StateGraph,END,START
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import sqlite3
load_dotenv()

True

In [2]:
llm_model = ChatGroq(model="llama-3.3-70b-versatile",temperature=0.3)

In [6]:
sqlite_conn = sqlite3.connect("sqlitesaver.sqlite",check_same_thread=False)
memory = SqliteSaver(conn = sqlite_conn)

In [9]:
class BasicChatState(TypedDict):
    messages: Annotated[list,add_messages]

def chatbot(state:BasicChatState):
    return {"messages":[llm_model.invoke(state["messages"])]}

graph = StateGraph(BasicChatState)

graph.add_node("agent",chatbot)
graph.add_edge(START,"agent")
graph.add_edge("agent",END)
workflow = graph.compile(checkpointer=memory)
config = {"configurable":{"thread_id":1}}
while True:
    user_input = input("User : ")
    if user_input in ["end","quit","exit"]:
        break
    else:
        result = workflow.invoke(input={"messages":[HumanMessage(content=user_input)]},config=config)
        print("AI : ",result["messages"][-1].content)
        

AI :  Your name is Hrishikesh.
AI :  Hello again, Hrishikesh. How can I assist you today?
AI :  Here's a thought for the day:

"Believe you can and you're halfway there." 

Remember, Hrishikesh, your thoughts have the power to shape your reality. So, choose to think positively, stay focused, and never give up on your dreams. You got this. Have a great day.
AI :  You're welcome, Hrishikesh. I hope the thought inspires you to tackle the day with confidence and positivity. If you need anything else, feel free to ask. Have a great day ahead.
